## Mean Curvature (UAV Lidar DEM)
### Mean curvature from second-order partial derivatives of the surface z(x,y).
### Positive = convex, Negative = concave. 

#### CRS: EPSG:25832.


In [ ]:
# USER CONFIGURATION 
DEM_PATH      = "data/dem.tif"
OUTPUT_PATH   = "output/mean_curvature.tif"
SMOOTH_SIGMA  = 1.0    # Gaussian pre-smoothing (in pixels); 0 to disable

In [ ]:
# Import required Libraries and read DEM file

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from dem_utils import read_dem, write_raster

dem, profile, (res_x, res_y) = read_dem(DEM_PATH)
print(f"DEM shape: {dem.shape}, pixel size: {res_x} x {res_y} m")

In [ ]:
# Compute mean curvature

def compute_mean_curvature(dem, res_x, res_y, sigma=1.0):
    """Mean curvature H of surface z(x,y).

    H = ((1+fx^2)*fyy - 2*fx*fy*fxy + (1+fy^2)*fxx) / (2*(1+fx^2+fy^2)^1.5)
    """
    mask = np.isnan(dem)
    z = np.where(mask, np.nanmean(dem), dem)

    if sigma and sigma > 0:
        z = ndimage.gaussian_filter(z, sigma=sigma)

    # First derivatives
    fy, fx = np.gradient(z, res_y, res_x)
    # Second derivatives
    fyy, fyx = np.gradient(fy, res_y, res_x)
    fxy, fxx = np.gradient(fx, res_y, res_x)

    denom = 2.0 * np.power(1.0 + fx**2 + fy**2, 1.5)
    numer = (1 + fx**2) * fyy - 2 * fx * fy * fxy + (1 + fy**2) * fxx
    H = numer / denom

    H[mask] = np.nan
    return H

mean_curv = compute_mean_curvature(dem, res_x, res_y, SMOOTH_SIGMA)

# Visualize result
# Symmetric color scale around zero
vmax = np.nanpercentile(np.abs(mean_curv), 98)
plt.figure(figsize=(9, 7))
im = plt.imshow(mean_curv, cmap="RdBu", vmin=-vmax, vmax=vmax)
plt.colorbar(im, label="Mean curvature [1/m]")
plt.title("Mean Curvature (blue=concave, red=convex)")
plt.axis("off")
plt.show()

In [ ]:
# Save mean curvature to output file
write_raster(OUTPUT_PATH, mean_curv, profile)